# LAB 04 — Multi-Source Data Ingestion with PySpark & Delta Lake
### Retail Customer 360 · Databricks Solutions Architect Interview Lab

**Platform:** Databricks Free Edition (Serverless) · **Language:** Python / PySpark / Spark SQL  
**Source:** Public GitHub repository `normanmartinez2/retail_customer360`, **all files under `data/crm/`**  
**Landing:** Unity Catalog managed Volume · **Target:** five managed Delta tables in `retail_customer360.bronze`  
**Prerequisites:** LAB 01 complete; LAB 02 `bronze.customers_raw` available. LAB 03 is helpful but not required for this ingestion notebook.  
**Important:** This notebook **does not** clean source data, modify customer tables, or process LAB 06 CDC feeds. Those are separate tasks.

> **Training goal:** Demonstrate how multiple systems can be ingested consistently while preserving source data, provenance, verifiable file versions, and repeatable execution. All walkthroughs and code comments are in English.

## Learning outcomes and demo storyline

A fictional retailer operates in Spain, Portugal, France, and Germany. Its CRM, ERP, e-commerce, payment gateway, and marketing platforms publish heterogeneous data. A unified analytical platform requires a repeatable approach to ingesting all datasets before enforcing shared business rules.

By the end of this lab, you should be able to:

1. Define **source contracts** and explicit schemas without relying on inference.
2. Download public, version-checked source files into a governed **Unity Catalog Volume**.
3. Load all five datasets with **PySpark** and persist managed **Delta** tables.
4. Attach technical provenance to every row and confirm that each batch is complete.
5. Inspect relationships and quality anomalies **without cleaning Bronze**.
6. Explain to a customer when GitHub-based ingestion is suitable for a demo versus production connectors / object storage / CDC.

**Logical data flow**

```text
GitHub (data/crm/*)                         Unity Catalog
   ├─ products_lab04_v1.csv       ──┐      retail_customer360
   ├─ orders_lab04_v1.csv         ──┤           ├─ bronze.customers_raw (LAB 02)
   ├─ order_items_lab04_v1.csv    ──┤ HTTPS     ├─ bronze.retail_source_files (Volume)
   ├─ payments_lab04_v1.csv       ──┤ → Volume  ├─ bronze.products_raw
   └─ customer_interactions_...  ──┘ → Spark   ├─ bronze.orders_raw
                                               ├─ bronze.order_items_raw
                                               ├─ bronze.payments_raw
                                               └─ bronze.customer_interactions_raw
                                               ↓
                                         LAB 05 — Silver Data Quality
                                         LAB 06 — Incremental CDC
                                         LAB 07 — Gold Customer 360
```

Note that the GitHub source folder is **flat** (`data/crm/`). The Volume will arrange files by logical source domain without moving or changing files in GitHub.

## STEP 01 — Verify the Databricks environment

**Purpose:** Point every query to explicit Unity Catalog objects and fail early if LAB 02 is not available. This notebook assumes a Serverless-compatible Unity Catalog workspace.

- Catalog: `retail_customer360`
- Schemas: `bronze`, `silver`, `gold`
- Required existing table: `retail_customer360.bronze.customers_raw`

The five new datasets are loaded to **Bronze**, not Silver. We intentionally preserve invalid business values so that LAB 05 can detect them.

In [ ]:
# STEP 01 — Configure fully qualified names and the source repository.
CATALOG = "retail_customer360"
BRONZE_SCHEMA = "bronze"
CUSTOMERS_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.customers_raw"

REPOSITORY = "normanmartinez2/retail_customer360"
REPOSITORY_URL = f"https://github.com/{REPOSITORY}"
RAW_BASE_URL = (
    "https://raw.githubusercontent.com/"
    f"{REPOSITORY}/refs/heads/main/data/crm"
)

# Use the same logical batch ID across all five source snapshots.
BATCH_ID = "LAB04_INITIAL_V1"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

assert spark.catalog.tableExists(CUSTOMERS_TABLE), (
    f"Required LAB 02 table not found: {CUSTOMERS_TABLE}. "
    "Finish LAB 02 before executing LAB 04."
)

print("Catalog:", spark.catalog.currentCatalog())
print("Schema:", spark.catalog.currentDatabase())
print("Existing CRM source:", CUSTOMERS_TABLE)
print("GitHub folder:", RAW_BASE_URL)


## STEP 02 — Declare source contracts and expected file identities

**Purpose:** Treat each CSV as an explicit, testable input instead of silently trusting whatever happens to be on the `main` branch.

For each file, the contract records its **column names and order**, expected **row count**, **SHA-256 of the original file bytes**, business-key column, and Bronze destination. These were derived from the prepared LAB 04 datasets and checked against their GitHub file versions.

A GitHub URL pointing at `main` is mutable. **SHA-256 verification makes this lab reproducible**: if someone changes a source file, the notebook stops rather than loading unexpected data.

| Dataset | Source | Expected rows | Destination |
|---|---|---:|---|
| Products | ERP | 500 | `bronze.products_raw` |
| Orders | E-commerce | 15,000 | `bronze.orders_raw` |
| Order items | E-commerce | 40,000 | `bronze.order_items_raw` |
| Payments | Payment gateway | 15,500 | `bronze.payments_raw` |
| Customer interactions | Marketing | 8,000 | `bronze.customer_interactions_raw` |
| **Total** | Five additional sources | **79,000** | Five Delta tables |

The existing CRM source remains untouched. LAB 06 CDC files are deliberately excluded.

In [ ]:
# STEP 02 — Explicit source contracts based on actual CSV headers.
# Preserve all source fields as strings in Bronze (including dates, amounts, and flags).
# Expected SHA-256 hashes refer to the original CSV bytes, not a transformed DataFrame.
DATASETS = [
    {
        "key": "products",
        "filename": "products_lab04_v1.csv",
        "domain": "erp",
        "bronze_table": "products_raw",
        "primary_key": "product_id",
        "expected_rows": 500,
        "sha256": "ddd4d8be71c1d009ea59a01ed57191b596fcd813fb152277628a5d542da5192d",
        "github_blob_sha": "3a537b955a61302489b5426b009dde5a055ed912",
        "columns": [
            "product_id",
            "sku",
            "product_name",
            "category",
            "subcategory",
            "brand",
            "unit_price_eur",
            "cost_price_eur",
            "supplier_id",
            "active_flag",
            "created_at",
            "updated_at",
            "source_system",
            "notes"
        ]
    },
    {
        "key": "orders",
        "filename": "orders_lab04_v1.csv",
        "domain": "ecommerce",
        "bronze_table": "orders_raw",
        "primary_key": "order_id",
        "expected_rows": 15000,
        "sha256": "f3a87d3e487dc956959240ac8ef383508bd6d28d069f2a087ef6325ce6658b2b",
        "github_blob_sha": "b2718182beb3975b898221ba60a420d7e3fbe3f4",
        "columns": [
            "order_id",
            "customer_id",
            "order_timestamp",
            "order_status",
            "sales_channel",
            "store_id",
            "country",
            "currency",
            "subtotal_eur",
            "discount_eur",
            "shipping_eur",
            "total_amount_eur",
            "updated_at",
            "source_system",
            "notes"
        ]
    },
    {
        "key": "order_items",
        "filename": "order_items_lab04_v1.csv",
        "domain": "ecommerce",
        "bronze_table": "order_items_raw",
        "primary_key": "order_item_id",
        "expected_rows": 40000,
        "sha256": "60b680dad2b9f9711acad7928f83bb5d191ecdbba6eb2a118178bfce6fcc41a3",
        "github_blob_sha": "5c721ec91964e03096859a5f7883baad307c2e8f",
        "columns": [
            "order_item_id",
            "order_id",
            "product_id",
            "quantity",
            "unit_price_eur",
            "gross_amount_eur",
            "line_discount_eur",
            "line_total_eur",
            "currency",
            "created_at",
            "updated_at",
            "source_system",
            "notes"
        ]
    },
    {
        "key": "payments",
        "filename": "payments_lab04_v1.csv",
        "domain": "payments",
        "bronze_table": "payments_raw",
        "primary_key": "payment_id",
        "expected_rows": 15500,
        "sha256": "df38b4e5587cdfc22206ed1841109744fbf15c4511bd4df24addaf84070b32a5",
        "github_blob_sha": "808c657eb0ec774ad8886fb9acb46e284b2db465",
        "columns": [
            "payment_id",
            "order_id",
            "customer_id",
            "payment_timestamp",
            "payment_type",
            "payment_status",
            "payment_method",
            "amount_eur",
            "currency",
            "gateway_reference",
            "updated_at",
            "source_system",
            "notes"
        ]
    },
    {
        "key": "customer_interactions",
        "filename": "customer_interactions_lab04_v1.csv",
        "domain": "marketing",
        "bronze_table": "customer_interactions_raw",
        "primary_key": "interaction_id",
        "expected_rows": 8000,
        "sha256": "fde689139efd0d2a7b67b871482f0519a78fec523ae098e35cb1f2be5b40a8d0",
        "github_blob_sha": "04f7c7d47530af2e756cf22a5a2b6369540f8124",
        "columns": [
            "interaction_id",
            "customer_id",
            "interaction_timestamp",
            "ingestion_timestamp",
            "channel",
            "event_type",
            "campaign_id",
            "interaction_outcome",
            "device_type",
            "country",
            "source_system",
            "notes"
        ]
    }
]

assert len(DATASETS) == 5
assert sum(spec["expected_rows"] for spec in DATASETS) == 79000

for spec in DATASETS:
    spec["source_url"] = f"{RAW_BASE_URL}/{spec['filename']}"
    spec["table_fqn"] = f"{CATALOG}.{BRONZE_SCHEMA}.{spec['bronze_table']}"

print("Source contracts ready:", len(DATASETS))
for spec in DATASETS:
    print(f"{spec['key']}: {spec['expected_rows']:,} rows → {spec['table_fqn']}")


## STEP 03 — Create the managed landing Volume

**Purpose:** Separate ingestion of file bytes from processing of CSV records.

The landing Volume is managed by Unity Catalog, whereas the Delta tables are managed tabular objects. The repository is flat, but we organize the landed files under `lab04/<logical-domain>/` so that their intended origin remains clear.

Creating the Volume requires appropriate privileges on the Bronze schema. If the Volume already exists, `IF NOT EXISTS` is a no-op.

In [ ]:
# STEP 03 — Create a managed UC Volume for all LAB 04 source CSVs.
VOLUME_NAME = "retail_source_files"
VOLUME_FQN = f"{CATALOG}.{BRONZE_SCHEMA}.{VOLUME_NAME}"
VOLUME_ROOT = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"

spark.sql(f"""
CREATE VOLUME IF NOT EXISTS {VOLUME_FQN}
COMMENT 'Version-verified source files for Retail Customer 360 labs'
""")

# Store source files in domain-specific subdirectories inside this single Volume.
for spec in DATASETS:
    spec["volume_directory"] = f"{VOLUME_ROOT}/lab04/{spec['domain']}"
    spec["volume_path"] = f"{spec['volume_directory']}/{spec['filename']}"
    dbutils.fs.mkdirs(spec["volume_directory"])

print("Landing Volume:", VOLUME_FQN)
for spec in DATASETS:
    print(spec["volume_path"])


## STEP 04 — Download, verify, and land the exact GitHub files

**Purpose:** Download source bytes via HTTPS and verify SHA-256 *before* Spark reads the CSV. Existing files are reused **only when their checksums match**.

**Execution environment caveat:** Public HTTPS access from Databricks Serverless can be restricted. If a download fails, use the manual fallback in the next section. No fabricated substitute data should be generated.

For ~5 MB files, a streamed Python download is sufficient for a demo. A production ingestion architecture would use approved connectors, cloud object storage, secrets, retries, orchestration, and operational monitoring.

In [ ]:
# STEP 04A — Declare reusable file hashing and download utilities.
import os
import hashlib
import urllib.request


def calculate_file_sha256(file_path, chunk_size=1024 * 1024):
    """Calculate a SHA-256 digest without loading the entire file into memory."""
    sha256 = hashlib.sha256()
    with open(file_path, "rb") as source_file:
        while True:
            block = source_file.read(chunk_size)
            if not block:
                break
            sha256.update(block)
    return sha256.hexdigest()


def land_dataset(spec):
    """Download a file from GitHub only if an exact matching local copy is absent."""
    destination = spec["volume_path"]
    expected_sha256 = spec["sha256"]

    # Never silently replace an existing file with unexpected content.
    if os.path.exists(destination):
        actual_sha256 = calculate_file_sha256(destination)
        if actual_sha256 != expected_sha256:
            raise ValueError(
                f"Existing Volume file differs from its contract: {destination}. "
                "Investigate the mismatch before continuing."
            )
        return "REUSED_AND_VERIFIED"

    request = urllib.request.Request(
        spec["source_url"],
        headers={"User-Agent": "RetailCustomer360-Databricks-LAB04/1.0"},
    )

    try:
        # Stream bytes directly to the governed Volume, preserving the CSV content.
        with urllib.request.urlopen(request, timeout=60) as response:
            with open(destination, "wb") as output_file:
                while True:
                    block = response.read(1024 * 1024)
                    if not block:
                        break
                    output_file.write(block)

        # Stop immediately if GitHub's mutable main branch no longer matches the lab.
        actual_sha256 = calculate_file_sha256(destination)
        if actual_sha256 != expected_sha256:
            raise ValueError(
                f"SHA-256 mismatch for {spec['filename']}. "
                "The GitHub content has changed or the download is incomplete."
            )

        return "DOWNLOADED_AND_VERIFIED"

    except Exception as exc:
        # Discard partial downloads; do not leave corrupt files in the landing area.
        if os.path.exists(destination):
            os.remove(destination)
        raise RuntimeError(
            f"Could not land {spec['filename']}. "
            f"Download the source manually from {spec['source_url']} "
            f"and upload the ORIGINAL file to {destination}; "
            "then rerun this cell. Original error: " + str(exc)
        ) from exc


In [ ]:
# STEP 04B — Download all five source CSVs into the UC Volume.
# The function reuses previously verified copies on subsequent notebook runs.
landing_results = []

for spec in DATASETS:
    status = land_dataset(spec)
    landing_results.append({
        "dataset": spec["key"],
        "file": spec["filename"],
        "status": status,
        "expected_sha256": spec["sha256"],
        "volume_path": spec["volume_path"],
    })
    print(f"[{status}] {spec['filename']}")

print("All five landed files passed SHA-256 verification.")


### Manual fallback — only if HTTPS is blocked

If STEP 04B fails for a file:

1. Open that file directly from `https://github.com/normanmartinez2/retail_customer360/tree/main/data/crm` and download its original CSV (not a copied text snippet).
2. Open **Catalog Explorer** → `retail_customer360` → `bronze` → **Volumes** → `retail_source_files`.
3. Upload it to the exact path displayed in STEP 03. If the required domain subdirectory is not shown, create it first in the Volume browser. Do **not** rename the file.
4. Rerun STEP 04B. It will verify the uploaded bytes using the expected SHA-256, then continue.

If a checksum differs, do not disable checksum validation. Confirm that you uploaded the exact source version.

## STEP 05 — Independently verify landing files and CSV headers

**Purpose:** Confirm all five files exist and contain exactly the expected bytes **before any tables are created**.

The following check is independent of Spark. It validates file presence, byte identity, and header order. The separate row-count verification will be performed by Spark when the files are read.

In [ ]:
# STEP 05 — Fail before table writes if any landed file or header is wrong.
import csv

for spec in DATASETS:
    path = spec["volume_path"]
    assert os.path.isfile(path), f"Missing landed file: {path}"
    assert calculate_file_sha256(path) == spec["sha256"], (
        f"Incorrect SHA-256: {spec['filename']}"
    )

    # Inspect only the CSV header, preserving its original spelling and column order.
    with open(path, "r", encoding="utf-8-sig", newline="") as csv_file:
        actual_header = next(csv.reader(csv_file))

    assert actual_header == spec["columns"], (
        f"Unexpected CSV header for {spec['filename']}: {actual_header}"
    )

    print(f"PASS: {spec['filename']} ({len(actual_header)} columns)")

print("Landing integrity and CSV contract checks passed.")


## STEP 06 — Build explicit all-STRING PySpark schemas

**Purpose:** Prevent schema inference from hiding source problems. Each CSV includes deliberate anomalies: malformed timestamps, duplicate IDs, invalid categories, invalid amounts, and missing references. Those anomalies must remain visible in Bronze.

**Why STRING in Bronze?** Amounts, flags, and timestamps are represented as raw source text here. LAB 05 will validate and convert them to `DECIMAL`, `BOOLEAN`, `DATE`, or `TIMESTAMP` using explicit rules.

The Marketing source contains a business-system `ingestion_timestamp` column. We preserve that original value as a STRING and add our separate technical `bronze_ingestion_timestamp` (TIMESTAMP), rather than accidentally overwriting it.

Be aware that the Spark CSV reader interprets the CSV format: empty fields may be read as `NULL`, and CSV quoting is parsed. Preserving *exact original bytes* is the responsibility of the landed file in the Volume, not the tabular Bronze representation.

In [ ]:
# STEP 06 — Build one reusable schema function for all five datasets.
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import functions as F


def source_schema(spec):
    """Construct a permissive string schema that exactly matches the CSV header."""
    return StructType([
        StructField(column_name, StringType(), nullable=True)
        for column_name in spec["columns"]
    ])


def load_bronze_dataframe(spec):
    """Parse a landed CSV and add consistent, auditable ingestion metadata."""
    source_df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("encoding", "UTF-8")
        .option("mode", "FAILFAST")  # Reject broken CSV rows, not invalid business values.
        .option("enforceSchema", "false")  # Check the header against the schema.
        .option("ignoreLeadingWhiteSpace", "false")
        .option("ignoreTrailingWhiteSpace", "false")
        .schema(source_schema(spec))
        .load(spec["volume_path"])
    )

    # These metadata columns are added without altering source business columns.
    return (
        source_df
        .withColumn("source_file", F.lit(spec["filename"]))
        .withColumn("source_file_url", F.lit(spec["source_url"]))
        .withColumn("source_repository", F.lit(REPOSITORY_URL))
        .withColumn("source_domain", F.lit(spec["domain"]))
        .withColumn("source_file_sha256", F.lit(spec["sha256"]))
        .withColumn("ingestion_batch_id", F.lit(BATCH_ID))
        .withColumn("bronze_ingestion_timestamp", F.current_timestamp())
        .withColumn("ingestion_layer", F.lit("BRONZE"))
    )

print("PySpark source schema and Bronze ingestion utilities ready.")


## STEP 07 — Preflight each source with Spark

**Purpose:** Verify that the CSV reader produces exactly the expected columns and row counts **before creating any Delta tables**. Since `FAILFAST` is enabled, structurally broken CSV records cause the read to fail instead of being silently discarded.

A record with an invalid email, status, timestamp, or monetary value is still a valid *CSV row* and must be preserved. Those are **business-quality issues**, not necessarily **CSV parsing errors**.

In [ ]:
# STEP 07 — Preflight source row counts and schemas with Spark.
preflight_results = []

for spec in DATASETS:
    df_source = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("encoding", "UTF-8")
        .option("mode", "FAILFAST")
        .option("enforceSchema", "false")
        .schema(source_schema(spec))
        .load(spec["volume_path"])
    )

    actual_count = df_source.count()
    actual_columns = df_source.columns

    assert actual_count == spec["expected_rows"], (
        f"Unexpected source record count for {spec['key']}: "
        f"{actual_count} vs {spec['expected_rows']}"
    )
    assert actual_columns == spec["columns"], (
        f"Unexpected columns for {spec['key']}"
    )

    preflight_results.append({
        "dataset": spec["key"],
        "rows": actual_count,
        "columns": len(actual_columns),
        "result": "PASS",
    })
    print(f"PASS: {spec['filename']} — {actual_count:,} rows")

assert sum(row["rows"] for row in preflight_results) == 79000
print("All 79,000 source rows are ready for Bronze ingestion.")


## STEP 08 — Write managed Delta tables in Bronze (safe initial load)

**Purpose:** Persist each independently verified source as a governed Delta table. We use **create-if-missing semantics** rather than `overwrite`:

- If the table does not exist, create it with `.format("delta").mode("errorifexists").saveAsTable(...)`.
- If the table already exists, validate its row count, original business schema, and source file SHA-256. **Never silently overwrite a previous Bronze table.**
- Do not use incremental `append` or `MERGE` here: that will be covered in LAB 06 with separate CDC files.

This structure makes rerunning the initial lab safe; if one table was created before a later cell failed, the successful tables remain intact.

In [ ]:
# STEP 08 — Initialize Bronze Delta tables without overwriting existing data.
INGESTION_METADATA_COLUMNS = [
    "source_file", "source_file_url", "source_repository", "source_domain",
    "source_file_sha256", "ingestion_batch_id", "bronze_ingestion_timestamp",
    "ingestion_layer",
]


def verify_existing_bronze_table(spec):
    """Verify that an already-created table represents the exact expected snapshot."""
    existing = spark.table(spec["table_fqn"])
    actual_count = existing.count()
    assert actual_count == spec["expected_rows"], (
        f"Existing table {spec['table_fqn']} has {actual_count} rows; "
        f"expected {spec['expected_rows']}. Do not overwrite without investigation."
    )
    assert existing.columns == spec["columns"] + INGESTION_METADATA_COLUMNS, (
        f"Schema mismatch in {spec['table_fqn']}. Review it manually."
    )
    file_hashes = [r[0] for r in existing.select("source_file_sha256").distinct().collect()]
    assert file_hashes == [spec["sha256"]], (
        f"Existing {spec['table_fqn']} has an unexpected source hash."
    )
    batch_ids = [r[0] for r in existing.select("ingestion_batch_id").distinct().collect()]
    assert batch_ids == [BATCH_ID], (
        f"Existing {spec['table_fqn']} was loaded by a different batch."
    )
    return actual_count


def initialize_bronze_table(spec):
    """Create a snapshot only once; otherwise validate the existing table."""
    if spark.catalog.tableExists(spec["table_fqn"]):
        count = verify_existing_bronze_table(spec)
        return "REUSED_VERIFIED", count

    df_bronze = load_bronze_dataframe(spec)
    actual_count = df_bronze.count()
    assert actual_count == spec["expected_rows"], (
        f"Source changed before Bronze write: {spec['key']}"
    )

    # This first-write-only operation preserves any existing table history.
    (
        df_bronze.write
        .format("delta")
        .mode("errorifexists")
        .saveAsTable(spec["table_fqn"])
    )

    # Read back the persistent Delta table to verify a successful commit.
    count = verify_existing_bronze_table(spec)
    return "CREATED", count


ingestion_results = []
for spec in DATASETS:
    status, row_count = initialize_bronze_table(spec)
    ingestion_results.append({
        "dataset": spec["key"],
        "delta_table": spec["table_fqn"],
        "rows": row_count,
        "status": status,
    })
    print(f"[{status}] {spec['table_fqn']}: {row_count:,} rows")

print("LAB 04 Bronze initial ingestion finished.")


## STEP 09 — Display an ingestion audit summary

**Purpose:** Provide a short, reproducible technical report that you can show during an architecture demo. The first run should show `CREATED`; later runs should show `REUSED_VERIFIED`.

The statuses are notebook output, not an enterprise audit log. A production pipeline should persist job-run details to monitoring or audit tables.

In [ ]:
# STEP 09 — Display source-to-Bronze lineage at a glance.
print("Expected total LAB 04 records:", sum(s["expected_rows"] for s in DATASETS))
print("Actual total LAB 04 records:", sum(r["rows"] for r in ingestion_results))

display(spark.createDataFrame(ingestion_results).orderBy("dataset"))
assert sum(r["rows"] for r in ingestion_results) == 79000


## STEP 10 — Explore the five Bronze tables

**Purpose:** Prove that Delta tables can be accessed using PySpark and SQL, independently of the source CSV files. Inspect the original string-typed business columns alongside the technical ingestion fields.

Try opening **Catalog Explorer → retail_customer360 → bronze** as well. The five new tables and the landing Volume should be visible, alongside `customers_raw` from LAB 02.

In [ ]:
# STEP 10A — Preview each newly created managed Delta table.
for spec in DATASETS:
    print("Dataset:", spec["key"], "| Delta table:", spec["table_fqn"])
    display(spark.table(spec["table_fqn"]).limit(5))


In [ ]:
# STEP 10B — Inspect a managed Delta table's metadata and transaction history.
products_fqn = f"{CATALOG}.{BRONZE_SCHEMA}.products_raw"
display(spark.sql(f"DESCRIBE DETAIL {products_fqn}"))
display(spark.sql(f"DESCRIBE HISTORY {products_fqn}"))


In [ ]:
# STEP 10C — List tables and Volumes inside the Bronze schema.
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{BRONZE_SCHEMA}"))
display(spark.sql(f"SHOW VOLUMES IN {CATALOG}.{BRONZE_SCHEMA}"))


## STEP 11 — Inspect data quality signals without modifying Bronze

**Purpose:** Verify that the synthetic edge cases survived ingestion. We deliberately **do not** drop duplicates, normalize invalid countries, fix amounts, or resolve foreign-key references in this lab.

Every physical row must remain available for LAB 05 quality rules and quarantine. The examples below are **exploratory checks only**, not the final enterprise data quality contract.

In [ ]:
# STEP 11A — Count null/blank business keys and duplicate-key groups.
# These checks do not remove or modify any records.
quality_overview = []
for spec in DATASETS:
    df = spark.table(spec["table_fqn"])
    key_col = spec["primary_key"]

    missing_keys = df.filter(
        F.col(key_col).isNull() | (F.trim(F.col(key_col)) == "")
    ).count()

    duplicate_key_groups = (
        df.filter(F.col(key_col).isNotNull() & (F.trim(F.col(key_col)) != ""))
        .groupBy(key_col)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    quality_overview.append({
        "dataset": spec["key"],
        "total_rows": df.count(),
        "missing_business_keys": missing_keys,
        "duplicate_key_groups": duplicate_key_groups,
    })

display(spark.createDataFrame(quality_overview).orderBy("dataset"))


### Why referential integrity cannot be assumed in Bronze

Our source files intentionally contain orders with customer IDs that do not exist in the CRM, order items referencing unknown products, and payments referencing missing orders. Some customer IDs may exist in **Bronze CRM** but fail **Silver CRM** validation.

The following left-anti joins quantify *unknown* references against raw keys. Missing foreign keys are counted separately. Remember that these are intentionally dirty, **unconformed source snapshots**, not a valid relational warehouse.

In [ ]:
# STEP 11B — Compare raw foreign-key relationships across systems.
# Distinct non-null business keys prevent source duplicates from multiplying matches.
customers = spark.table(CUSTOMERS_TABLE)
products = spark.table(f"{CATALOG}.bronze.products_raw")
orders = spark.table(f"{CATALOG}.bronze.orders_raw")
items = spark.table(f"{CATALOG}.bronze.order_items_raw")
payments = spark.table(f"{CATALOG}.bronze.payments_raw")
interactions = spark.table(f"{CATALOG}.bronze.customer_interactions_raw")

crm_keys = customers.select("customer_id").filter(
    F.col("customer_id").isNotNull() & (F.trim("customer_id") != "")
).distinct()
order_keys = orders.select("order_id").filter(
    F.col("order_id").isNotNull() & (F.trim("order_id") != "")
).distinct()
product_keys = products.select("product_id").filter(
    F.col("product_id").isNotNull() & (F.trim("product_id") != "")
).distinct()


def count_missing_reference(df, key):
    return df.filter(F.col(key).isNull() | (F.trim(F.col(key)) == "")).count()


def count_unknown_reference(df, key, reference_df):
    return (
        df.filter(F.col(key).isNotNull() & (F.trim(F.col(key)) != ""))
        .join(reference_df, on=key, how="left_anti")
        .count()
    )

reference_checks = [
    ("orders → CRM customers", orders, "customer_id", crm_keys),
    ("items → orders", items, "order_id", order_keys),
    ("items → products", items, "product_id", product_keys),
    ("payments → orders", payments, "order_id", order_keys),
    ("interactions → CRM customers", interactions, "customer_id", crm_keys),
]

reference_results = []
for relationship, source_df, fk_col, reference_df in reference_checks:
    reference_results.append({
        "relationship": relationship,
        "missing_fk_rows": count_missing_reference(source_df, fk_col),
        "unknown_fk_rows": count_unknown_reference(source_df, fk_col, reference_df),
    })

display(spark.createDataFrame(reference_results))


In [ ]:
# STEP 11C — Explore a few examples of malformed values without repairing them.
# These checks intentionally operate on raw STRING fields.
print("Order statuses present in Bronze:")
display(orders.groupBy("order_status").count().orderBy(F.desc("count")))

print("Payment statuses present in Bronze:")
display(payments.groupBy("payment_status").count().orderBy(F.desc("count")))

print("Example order timestamps that cannot be converted safely:")
display(
    orders.filter(
        F.col("order_timestamp").isNotNull()
        & F.expr("try_cast(order_timestamp AS TIMESTAMP) IS NULL")
    ).select("order_id", "order_timestamp", "source_file").limit(15)
)

print("Example order totals that cannot be parsed as decimal:")
display(
    orders.filter(
        F.col("total_amount_eur").isNotNull()
        & F.expr("try_cast(total_amount_eur AS DECIMAL(18,2)) IS NULL")
    ).select("order_id", "total_amount_eur").limit(15)
)


## STEP 12 — Validate row counts, provenance, table formats, and rerun safety

**Purpose:** Turn the expected inputs into executable assertions. Counts and SHA-256 values come from the prepared dataset manifest; the checks query persisted **Delta** tables rather than local CSVs.

Remember: a row-count reconciliation proves completeness of the ingested snapshot, **not** its business correctness. LAB 05 will add business-level quality metrics.

In [ ]:
# STEP 12 — Verify all five Delta tables and their lineage metadata.
validated_rows = 0

for spec in DATASETS:
    df = spark.table(spec["table_fqn"])
    table_count = df.count()
    validated_rows += table_count

    assert table_count == spec["expected_rows"], (
        f"Incorrect row count in {spec['table_fqn']}"
    )

    # Check the table schema and the constant per-file ingestion fields.
    assert df.columns == spec["columns"] + INGESTION_METADATA_COLUMNS
    assert df.filter(F.col("source_file") != spec["filename"]).count() == 0
    assert df.filter(F.col("source_file_sha256") != spec["sha256"]).count() == 0
    assert df.filter(F.col("ingestion_batch_id") != BATCH_ID).count() == 0
    assert df.filter(F.col("bronze_ingestion_timestamp").isNull()).count() == 0

    # Delta's physical format is verified via catalog metadata.
    detail_row = spark.sql(f"DESCRIBE DETAIL {spec['table_fqn']}").select("format").first()
    assert detail_row[0].lower() == "delta"

    print(f"PASS: {spec['table_fqn']} — {table_count:,} rows, Delta format, verified origin")

assert validated_rows == 79000, (
    f"Expected 79,000 additional rows, found {validated_rows}"
)

print("LAB 04 FINAL VALIDATION PASSED")


## STEP 13 — SQL demo: business context across independent source systems

Use this **read-only** exploratory query to show a cross-source analytical use case. We count order records by their country code as received. **Do not call the result certified revenue**, because the data has not yet passed quality checks, payment reconciliation, or currency rules.

This is an effective transition to LAB 05: we can access all the data, but we do not yet have a trusted business metric.

In [ ]:
# STEP 13 — Source-level order activity (unclean, illustrative only).
display(spark.sql(f"""
    SELECT
        country,
        COUNT(*) AS raw_order_rows,
        COUNT(DISTINCT order_id) AS distinct_order_ids,
        COUNT(DISTINCT customer_id) AS distinct_customer_ids
    FROM {CATALOG}.bronze.orders_raw
    GROUP BY country
    ORDER BY raw_order_rows DESC
"""))


## STEP 14 — Final architecture checkpoint

Verify that your workspace contains the following objects:

```text
retail_customer360
├── bronze
│   ├── customers_raw                  (from LAB 02 — unchanged)
│   ├── products_raw                   (500 rows)
│   ├── orders_raw                     (15,000 rows)
│   ├── order_items_raw                (40,000 rows)
│   ├── payments_raw                   (15,500 rows)
│   ├── customer_interactions_raw      (8,000 rows)
│   └── retail_source_files            (Unity Catalog Volume)
│       └── lab04
│           ├── erp/products_lab04_v1.csv
│           ├── ecommerce/orders_lab04_v1.csv
│           ├── ecommerce/order_items_lab04_v1.csv
│           ├── payments/payments_lab04_v1.csv
│           └── marketing/customer_interactions_lab04_v1.csv
├── silver                           (from LAB 03; untouched by LAB 04)
└── gold                             (for future labs)
```

**Successful completion criteria:** five valid Delta tables, exactly 79,000 ingested rows across them, matching SHA-256 provenance, and visible quality anomalies available for LAB 05. No LAB 06 CDC feeds are ingested yet.

## STEP 15 — Interview readiness: concise Solutions Architect answers

**Q1. Why use a shared Bronze ingestion pattern for different source systems?**  
**A.** A reusable contract-driven ingestion pattern provides consistent schema handling, observability, governance, and provenance. Each source still retains its own semantics, and the Silver layer applies domain-specific quality rules.

**Q2. Why read CSVs as strings instead of inferring types?**  
**A.** The sources intentionally contain malformed monetary amounts and timestamps. Ingestion should preserve those values for investigation. We apply explicit parsing and business-quality contracts when publishing Silver tables.

**Q3. Why use a Unity Catalog Volume before writing Delta tables?**  
**A.** The Volume governs the original files and separates file transport from parsing. The Delta tables provide a transactionally reliable representation for downstream data processing.

**Q4. How do we make the initial ingest repeatable?**  
**A.** We verify expected SHA-256 checksums, exact CSV headers and record counts. We create each table only if missing, and verify its identity when it already exists. We never append the same snapshot twice.

**Q5. Why not use GitHub for a production CRM / ERP ingestion pipeline?**  
**A.** GitHub is a transparent way to distribute reproducible synthetic lab data. A real deployment would use the approved source connector, ingestion landing storage, CDC or APIs as needed, plus authentication, network policies, retries, and monitoring.

**Q6. Do these five tables already form a trusted Customer 360?**  
**A.** No. Bronze deliberately preserves duplicates, unknown foreign keys, suspicious amounts, and invalid timestamps. LAB 05 will create cleaned and conformed Silver datasets, and LAB 07 will build business-ready Gold products.

**Q7. How would you explain Delta and Unity Catalog to a customer?**  
**A.** Delta Lake provides ACID table transactions and versioning, while Unity Catalog provides centralized data governance, object organization, and permissions. They address different aspects of the architecture and work together.

## Official study links

- [PySpark in Databricks](https://docs.databricks.com/aws/en/pyspark/)
- [Work with CSV files](https://docs.databricks.com/aws/en/query/formats/csv)
- [Unity Catalog Volumes](https://docs.databricks.com/aws/en/volumes/)
- [Managed tables](https://docs.databricks.com/aws/en/tables/managed)
- [Delta Lake](https://docs.databricks.com/aws/en/delta/)
- [Medallion Architecture](https://docs.databricks.com/aws/en/lakehouse/medallion)
- [Databricks Free Edition limitations](https://docs.databricks.com/aws/en/getting-started/free-edition-limitations)

**Next lab — LAB 05: Advanced Data Quality.** We will parse currencies, amounts, dates and flags; validate keys and business rules across systems; quarantine invalid rows; reconcile order totals, line items, and payment events; and publish conformed Silver tables without silently losing data.

**Future lab — LAB 06: Delta MERGE & CDC.** We will ingest the three existing CDC files from GitHub separately, implement deduplication and update strategies, and evaluate incremental SCD1/SCD2 patterns and out-of-order events.